In [36]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/DWDM_Project/data/processed/samsung_processed_dataset.csv")

df.head()

,Product,Category,Rating,Review,Platform,Date,Issue,RSI,Sentiment_score
0,Samsung Buds Live,Accessories,5,worth it house save decide although bag fight ...,Flipkart,2024-12-14,none,0.0600,0.5859
1,Samsung Galaxy M14,Smartphone,2,waste of money issue display flickering,Flipkart,2022-07-04,display,0.6536,-0.4215
2,Samsung Galaxy M53,Smartphone,5,excellent game threat see head cover task see,Flipkart,2022-07-21,none,0.0600,0.0772
3,Samsung Galaxy Tab A8,Smartphone,4,good,Amazon,2024-03-13,none,0.1350,0.4404
4,Samsung Charger 25W,Accessories,1,average join hotel daughter water,Flipkart,2024-11-13,heating,0.5600,0.2960


In [37]:
fact_review = df.copy()

fact_review["Review_ID"] = range(1, len(df)+1)

fact_review = fact_review[[
    "Review_ID",
    "Product",
    "Category",
    "Rating",
    "Sentiment_score",
    "Issue",
    "RSI",
    "Platform",
    "Date"
]]

fact_review.head()

,Review_ID,Product,Category,Rating,Sentiment_score,Issue,RSI,Platform,Date
0,1,Samsung Buds Live,Accessories,5,0.5859,none,0.0600,Flipkart,2024-12-14
1,2,Samsung Galaxy M14,Smartphone,2,-0.4215,display,0.6536,Flipkart,2022-07-04
2,3,Samsung Galaxy M53,Smartphone,5,0.0772,none,0.0600,Flipkart,2022-07-21
3,4,Samsung Galaxy Tab A8,Smartphone,4,0.4404,none,0.1350,Amazon,2024-03-13
4,5,Samsung Charger 25W,Accessories,1,0.2960,heating,0.5600,Flipkart,2024-11-13


 STEP 3 — Create Dimension Tables

In [38]:
dim_product = df[["Product", "Category"]].drop_duplicates().reset_index(drop=True)

dim_product["Product_ID"] = range(1, len(dim_product)+1)

dim_product.head()

,Product,Category,Product_ID
0,Samsung Buds Live,Accessories,1
1,Samsung Galaxy M14,Smartphone,2
2,Samsung Galaxy M53,Smartphone,3
3,Samsung Galaxy Tab A8,Smartphone,4
4,Samsung Charger 25W,Accessories,5


In [39]:
dim_platform = df[["Platform"]].drop_duplicates().reset_index(drop=True)

dim_platform["Platform_ID"] = range(1, len(dim_platform)+1)

dim_platform.head()

,Platform,Platform_ID
0,Flipkart,1
1,Amazon,2


In [40]:
dim_issue = df[["Issue"]].drop_duplicates().reset_index(drop=True)

dim_issue["Issue_ID"] = range(1, len(dim_issue)+1)

dim_issue.head(11)

,Issue,Issue_ID
0,none,1
1,display,2
2,heating,3
3,general_issue,4
4,software,5
5,battery,6
6,camera,7
7,build_quality,8
8,performance,9
9,sound,10


In [41]:
df["Date"] = pd.to_datetime(df["Date"])

dim_date = df[["Date"]].drop_duplicates().reset_index(drop=True)

dim_date["Date_ID"] = range(1, len(dim_date)+1)
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["Day"] = dim_date["Date"].dt.day

dim_date.head()

,Date,Date_ID,Year,Month,Day
0,2024-12-14,1,2024,12,14
1,2022-07-04,2,2022,7,4
2,2022-07-21,3,2022,7,21
3,2024-03-13,4,2024,3,13
4,2024-11-13,5,2024,11,13


STEP 4 — Replace with IDs

Merge Product_ID

In [42]:
fact_review = fact_review.merge(dim_product, on=["Product", "Category"], how="left")

Merge Platform_ID

In [43]:
fact_review = fact_review.merge(dim_platform, on="Platform", how="left")

Merge Issue_ID

In [44]:
fact_review = fact_review.merge(dim_issue, on="Issue", how="left")

Merge Date_ID

In [45]:
fact_review["Date"] = pd.to_datetime(fact_review["Date"])
fact_review = fact_review.merge(dim_date, on="Date", how="left")

STEP 5 — Clean Fact Table

In [46]:
fact_review = fact_review[[
    "Review_ID",
    "Product_ID",
    "Platform_ID",
    "Issue_ID",
    "Date_ID",
    "Rating",
    "Sentiment_score",
    "RSI"
]]

STEP 6 — Save all tables

In [47]:
base = "/content/drive/MyDrive/DWDM_Project/data/warehouse/"

fact_review.to_csv(base + "fact_review.csv", index=False)
dim_product.to_csv(base + "dim_product.csv", index=False)
dim_platform.to_csv(base + "dim_platform.csv", index=False)
dim_issue.to_csv(base + "dim_issue.csv", index=False)
dim_date.to_csv(base + "dim_date.csv", index=False)

print("✅ Warehouse tables saved")

✅ Warehouse tables saved


In [49]:
print(fact_review.head())
print(dim_date.head())

   Review_ID  Product_ID  Platform_ID  Issue_ID  Date_ID  Rating  \
0          1           1            1         1        1       5   
1          2           2            1         2        2       2   
2          3           3            1         1        3       5   
3          4           4            2         1        4       4   
4          5           5            1         3        5       1   

   Sentiment_score     RSI  
0           0.5859  0.0600  
1          -0.4215  0.6536  
2           0.0772  0.0600  
3           0.4404  0.1350  
4           0.2960  0.5600  
        Date  Date_ID  Year  Month  Day
0 2024-12-14        1  2024     12   14
1 2022-07-04        2  2022      7    4
2 2022-07-21        3  2022      7   21
3 2024-03-13        4  2024      3   13
4 2024-11-13        5  2024     11   13
